In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import pandas as pd
import numpy as np

# ✅ Your folder (as provided)
DATA_DIR = "/content/drive/My Drive/Colab Notebooks/Order_Forecasting/Input"

# Where to write the shifted (recent timeline) files
OUT_DIR = os.path.join(DATA_DIR, "shifted_recent")
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_MAX_DATE = pd.Timestamp.today().normalize()
PRESERVE_WEEKDAY = True  # shift by multiple of 7 days to keep weekday patterns aligned


def parse_dates_if_present(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    return df


def max_date_across(dfs, date_cols):
    max_dates = []
    for name, df in dfs.items():
        for c in date_cols:
            if c in df.columns:
                m = pd.to_datetime(df[c], errors="coerce").max()
                if pd.notna(m):
                    max_dates.append(m)
    if not max_dates:
        raise ValueError("No recognized date columns found in the loaded files.")
    return max(max_dates)


def shift_timeline(dfs, target_max_date, preserve_weekday=True):
    # supports Kaggle + your schema tables
    date_cols = [
        "date",
        "order_date",
        "inventory_date",
        "promo_start_date",
        "promo_end_date",
        "activity_date",
        "weather_date",
    ]

    # parse date columns
    parsed = {k: parse_dates_if_present(v.copy(), date_cols) for k, v in dfs.items()}

    current_max = max_date_across(parsed, date_cols)
    raw_delta = target_max_date.normalize() - current_max.normalize()

    if preserve_weekday:
        delta_days = raw_delta.days - (raw_delta.days % 7)
        delta = pd.Timedelta(days=delta_days)
    else:
        delta = raw_delta

    shifted = {}
    for name, df in parsed.items():
        d = df.copy()
        for c in date_cols:
            if c in d.columns:
                d[c] = pd.to_datetime(d[c], errors="coerce") + delta

        # If there's a calendar table, recompute derived fields (optional)
        if name.lower().startswith("calendar") and "date" in d.columns:
            # create common derived columns if they exist / or add them
            d["day_of_week"] = d["date"].dt.dayofweek + 1  # 1=Mon..7=Sun
            d["week_of_year"] = d["date"].dt.isocalendar().week.astype(int)
            d["month"] = d["date"].dt.month
            d["quarter"] = d["date"].dt.quarter
            d["year"] = d["date"].dt.year
            d["is_weekend"] = (d["date"].dt.dayofweek >= 5).astype(bool)

        shifted[name] = d

    info = {
        "current_max_date": current_max,
        "target_max_date": target_max_date.normalize(),
        "applied_delta_days": delta.days,
        "preserve_weekday": preserve_weekday,
    }
    return shifted, info


# --------------------------
# Load files if they exist
# --------------------------
# Kaggle Store Sales files (common names)
candidate_files = {
    "train": "train.csv",
    "test": "test.csv",
    "stores": "stores.csv",
    "holidays_events": "holidays_events.csv",
    "oil": "oil.csv",
    "sample_submission": "sample_submission.csv",
}

dfs = {}
for key, fname in candidate_files.items():
    fpath = os.path.join(DATA_DIR, fname)
    if os.path.exists(fpath):
        dfs[key] = pd.read_csv(fpath)
        print(f"Loaded: {fname}  shape={dfs[key].shape}")
    else:
        print(f"Missing (ok): {fname}")

if not dfs:
    raise FileNotFoundError(f"No CSV files found in: {DATA_DIR}")

# --------------------------
# Shift timeline to recent
# --------------------------
shifted_dfs, shift_info = shift_timeline(dfs, TARGET_MAX_DATE, PRESERVE_WEEKDAY)
print("\nShift info:", shift_info)

# quick sanity check for each table/file
for name, df in shifted_dfs.items():
    if "date" in df.columns:
        print(f"{name:15s} date min/max: {df['date'].min()} -> {df['date'].max()}")

# --------------------------
# Save shifted versions
# --------------------------
for name, df in shifted_dfs.items():
    out_path = os.path.join(OUT_DIR, f"{candidate_files[name].replace('.csv','')}_shifted.csv")
    df.to_csv(out_path, index=False)
    print("Saved:", out_path)

print("\n✅ Done. Shifted files are in:", OUT_DIR)

Loaded: train.csv  shape=(3000888, 6)
Loaded: test.csv  shape=(28512, 5)
Loaded: stores.csv  shape=(54, 5)
Loaded: holidays_events.csv  shape=(350, 6)
Loaded: oil.csv  shape=(1218, 2)
Loaded: sample_submission.csv  shape=(28512, 2)

Shift info: {'current_max_date': Timestamp('2017-12-26 00:00:00'), 'target_max_date': Timestamp('2026-03-01 00:00:00'), 'applied_delta_days': 2982, 'preserve_weekday': True}
train           date min/max: 2021-03-02 00:00:00 -> 2025-10-14 00:00:00
test            date min/max: 2025-10-15 00:00:00 -> 2025-10-30 00:00:00
holidays_events date min/max: 2020-05-01 00:00:00 -> 2026-02-24 00:00:00
oil             date min/max: 2021-03-02 00:00:00 -> 2025-10-30 00:00:00
Saved: /content/drive/My Drive/Colab Notebooks/Order_Forecasting/Input/shifted_recent/train_shifted.csv
Saved: /content/drive/My Drive/Colab Notebooks/Order_Forecasting/Input/shifted_recent/test_shifted.csv
Saved: /content/drive/My Drive/Colab Notebooks/Order_Forecasting/Input/shifted_recent/stores_s